# Data inspection — phys-tir / phys-cot

Visual spot-check of the merged training parquets after the corpus-rebuild (dropped OlympiadBench / SciBench-RL / PHYBench / PHYSICS).

**Expected results:**
- Only source = `UGPhysics` in the `curated` pool
- `drsci` rows have source = null (Dr. SCI doesn't use the per-source tag)
- Same user messages in TIR and CoT for any given row (only system prompt differs)
- Row counts: train 107,977 / val 2,200 / test 2,200

**Requirements:** needs `pyarrow` in your env. If you hit `ImportError`, run `pip install pyarrow` in this kernel's env.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_colwidth', 300)
pd.set_option('display.width', 200)

REPO = Path('..').resolve() if Path('notebooks').exists() is False else Path('.').resolve()
# Fall back to repo root if the notebook was opened from a different cwd
if not (REPO / 'data' / 'processed_tir').exists():
    REPO = Path('/diskarray/home/xs272/phys-reasoner')
TIR_DIR = REPO / 'data' / 'processed_tir' / 'data'
COT_DIR = REPO / 'data' / 'processed_cot' / 'data'
print('TIR dir:', TIR_DIR, 'exists:', TIR_DIR.exists())
print('CoT dir:', COT_DIR, 'exists:', COT_DIR.exists())

TIR dir: /diskarray/home/xs272/phys-reasoner/data/processed_tir/data exists: True
CoT dir: /diskarray/home/xs272/phys-reasoner/data/processed_cot/data exists: True


## 1. Load splits

In [2]:
tir = {s: pd.read_parquet(TIR_DIR / f'{s}.parquet') for s in ['train', 'validation', 'test']}
cot = {s: pd.read_parquet(COT_DIR / f'{s}.parquet') for s in ['train', 'validation', 'test']}

print('TIR sizes:', {s: len(df) for s, df in tir.items()})
print('CoT sizes:', {s: len(df) for s, df in cot.items()})
print('Columns   :', list(tir['train'].columns))

TIR sizes: {'train': 107977, 'validation': 2200, 'test': 2200}
CoT sizes: {'train': 107977, 'validation': 2200, 'test': 2200}
Columns   : ['data_source', 'prompt', 'reward_model', 'extra_info', 'pool']


## 2. Pool and source distributions

Expect only `UGPhysics` in the `curated` pool; `drsci` rows have null source.

In [3]:
def pool_source_table(df):
    sources = df['extra_info'].apply(lambda x: x.get('source') if isinstance(x, dict) else None)
    return pd.crosstab(df['pool'], sources.fillna('<null>'), margins=True, margins_name='total')

for split in ['train', 'validation', 'test']:
    print(f'\n=== TIR {split} ===')
    print(pool_source_table(tir[split]))


=== TIR train ===
extra_info  <null>  UGPhysics   total
pool                                 
curated          0       5414    5414
drsci       102563          0  102563
total       102563       5414  107977

=== TIR validation ===
extra_info  <null>  UGPhysics  total
pool                                
curated          0        200    200
drsci         2000          0   2000
total         2000        200   2200

=== TIR test ===
extra_info  <null>  UGPhysics  total
pool                                
curated          0        200    200
drsci         2000          0   2000
total         2000        200   2200


## 3. Contamination check (should all print CLEAN)

In [4]:
DROP = {'PHYSICS', 'OlympiadBench', 'SciBench_RL', 'PHYBench'}
for name, bundle in [('TIR', tir), ('CoT', cot)]:
    for split, df in bundle.items():
        sources = set(df['extra_info'].apply(lambda x: x.get('source')).dropna().unique())
        leaked = sources & DROP
        status = f'LEAK: {leaked}' if leaked else 'CLEAN'
        print(f'  {name} {split:12s}: sources={sources}  {status}')

  TIR train       : sources={'UGPhysics'}  CLEAN
  TIR validation  : sources={'UGPhysics'}  CLEAN
  TIR test        : sources={'UGPhysics'}  CLEAN
  CoT train       : sources={'UGPhysics'}  CLEAN
  CoT validation  : sources={'UGPhysics'}  CLEAN
  CoT test        : sources={'UGPhysics'}  CLEAN


## 4. Answer-type and domain distributions (curated pool only)

Dr. SCI uses different keys (`answer_type` string with its own vocabulary), so we focus on `curated` rows here.

In [5]:
curated_train = tir['train'][tir['train']['pool'] == 'curated'].copy()
curated_train['_pat'] = curated_train['extra_info'].apply(lambda x: x.get('primary_answer_type'))
curated_train['_dc']  = curated_train['extra_info'].apply(lambda x: x.get('domain_coarse'))

print('primary_answer_type:')
print(curated_train['_pat'].value_counts())
print('\ndomain_coarse:')
print(curated_train['_dc'].value_counts())
print('\npat x domain_coarse:')
print(pd.crosstab(curated_train['_pat'], curated_train['_dc'], margins=True))

primary_answer_type:
_pat
numerical     1998
expression    1581
equation       747
multi-part     672
mcq            208
true_false     146
interval        62
Name: count, dtype: int64

domain_coarse:
_dc
quantum_modern    1884
mechanics         1147
thermo_stat        915
em_electro         568
other              549
optics             351
Name: count, dtype: int64

pat x domain_coarse:
_dc         em_electro  mechanics  optics  other  quantum_modern  thermo_stat   All
_pat                                                                               
equation            73        254      32     58             170          160   747
expression         222        333      60    136             462          368  1581
interval             4         12       5      5              30            6    62
mcq                 25          0       0     27             156            0   208
multi-part          71        154      39     67             245           96   672
numerical          16

## 5. Dr. SCI: answer_type and `from` distributions

In [6]:
drsci_train = tir['train'][tir['train']['pool'] == 'drsci'].copy()
drsci_train['_at']   = drsci_train['extra_info'].apply(lambda x: x.get('answer_type'))
drsci_train['_from'] = drsci_train['extra_info'].apply(lambda x: x.get('from'))

print('answer_type:')
print(drsci_train['_at'].value_counts())
print("\nfrom (top 15):")
print(drsci_train['_from'].value_counts().head(15))

answer_type:
_at
equation      36876
numerical     23290
expression    21545
mcq           20852
Name: count, dtype: int64

from (top 15):
_from
MegaScience             55062
natural_reasoning       38784
WebInstruct-Verified     8717
Name: count, dtype: int64


## 6. System prompts — TIR vs CoT

Each dataset should have exactly ONE unique system prompt.

In [7]:
def unique_sys_prompts(df):
    return df['prompt'].apply(lambda msgs: msgs[0]['content']).unique()

tir_sys = unique_sys_prompts(tir['train'])
cot_sys = unique_sys_prompts(cot['train'])
print(f'TIR has {len(tir_sys)} unique system prompt(s)')
print(f'CoT has {len(cot_sys)} unique system prompt(s)')

print('\n=== TIR system prompt ===')
print(tir_sys[0])
print('\n=== CoT system prompt ===')
print(cot_sys[0])

TIR has 1 unique system prompt(s)
CoT has 1 unique system prompt(s)

=== TIR system prompt ===
You are an expert physics problem solver.
You have access to a Python interpreter. Use it when it helps — for numerical computation, symbolic algebra, or unit conversion. You are not required to use it.
Only call the tool if the code performs actual computation — numerical evaluation, symbolic solving, or unit conversion. Do not use it to print a formula or expression you derived in text.

Allowed packages: numpy, scipy, sympy, pint, math, cmath, statistics, fractions, decimal, itertools, functools, collections, random, re, json, io, typing, time
If you write code, it must call print() to output the result. Define all variables inside the code block.

You may execute code at most once. Once you have the result, give your final answer directly.

End with your final answer as \boxed{<value>}.


=== CoT system prompt ===
You are an expert physics problem solver.
Solve the problem step by step. S

## 7. TIR vs CoT row parity

Every row's user message, ground truth, and extra_info should be identical between TIR and CoT. Only the system prompt differs.

In [8]:
def user_text(msgs):
    for m in msgs:
        if m['role'] == 'user':
            return m['content']
    return ''

for split in ['train', 'validation', 'test']:
    t_user = tir[split]['prompt'].apply(user_text)
    c_user = cot[split]['prompt'].apply(user_text)
    user_match = (t_user == c_user).sum()
    t_gt = tir[split]['reward_model'].apply(lambda x: x['ground_truth'])
    c_gt = cot[split]['reward_model'].apply(lambda x: x['ground_truth'])
    gt_match = (t_gt == c_gt).sum()
    n = len(tir[split])
    print(f'  {split:12s}: user_match={user_match}/{n}  gt_match={gt_match}/{n}')

  train       : user_match=107977/107977  gt_match=107977/107977
  validation  : user_match=2200/2200  gt_match=2200/2200
  test        : user_match=2200/2200  gt_match=2200/2200


## 8. Random sample inspection

Eyeball 5 random rows from each pool — verify the prompt + answer look reasonable.

In [9]:
def show_row(row, label=''):
    print(f'--- {label} ---')
    ei = row['extra_info']
    print(f'  pool         : {row["pool"]}')
    print(f'  data_source  : {row["data_source"]}')
    print(f'  source       : {ei.get("source")}')
    print(f'  answer_type  : {ei.get("answer_type")}')
    print(f'  primary_type : {ei.get("primary_answer_type")}')
    print(f'  from         : {ei.get("from")}')
    print(f'  domain_coarse: {ei.get("domain_coarse")}')
    print(f'  ground_truth : {row["reward_model"]["ground_truth"][:150]}')
    user = next(m['content'] for m in row['prompt'] if m['role'] == 'user')
    print(f'  user (first 400 chars):')
    print(f'    {user[:400]}')
    print()

print('=' * 70)
print('CURATED SAMPLES (UGPhysics)')
print('=' * 70)
curated_samples = tir['train'][tir['train']['pool'] == 'curated'].sample(5, random_state=7)
for i, (_, row) in enumerate(curated_samples.iterrows()):
    show_row(row, f'curated {i+1}')

print('=' * 70)
print('DRSCI SAMPLES')
print('=' * 70)
drsci_samples = tir['train'][tir['train']['pool'] == 'drsci'].sample(5, random_state=7)
for i, (_, row) in enumerate(drsci_samples.iterrows()):
    show_row(row, f'drsci {i+1}')

CURATED SAMPLES (UGPhysics)
--- curated 1 ---
  pool         : curated
  data_source  : UGPhysics
  source       : UGPhysics
  answer_type  : numerical
  primary_type : numerical
  from         : None
  domain_coarse: thermo_stat
  ground_truth : \boxed{2 \gamma k T}
  user (first 400 chars):
    If a macroscopic small object is placed in a fluid with temperature \(T\), prove that due to the collisions with fluid molecules, the small object will experience a damping force: \(-\gamma v\) (where \(\gamma\) is the viscosity) and a random force \(f(t)\), such that the equation of motion is:

\[ m \frac{\mathrm{d} v}{\mathrm{~d} t}=-\gamma v+f(t) \]

where \(\overline{f(t)}=0\), \(\overline{f(t

--- curated 2 ---
  pool         : curated
  data_source  : UGPhysics
  source       : UGPhysics
  answer_type  : numerical
  primary_type : numerical
  from         : None
  domain_coarse: quantum_modern
  ground_truth : \boxed{1077}
  user (first 400 chars):
    The $\rho^{0}$ meson is a meson res

## 9. Side-by-side TIR vs CoT for the same row

Pick row index 0 of the train split in both datasets. User message and ground truth must match byte-for-byte. System prompt will differ.

In [10]:
idx = 0
t_row = tir['train'].iloc[idx]
c_row = cot['train'].iloc[idx]

print('User message equal :', next(m['content'] for m in t_row['prompt'] if m['role'] == 'user') == next(m['content'] for m in c_row['prompt'] if m['role'] == 'user'))
print('Ground truth equal :', t_row['reward_model']['ground_truth'] == c_row['reward_model']['ground_truth'])
print('Pool equal         :', t_row['pool'] == c_row['pool'])

print('\n--- TIR system ---')
print(t_row['prompt'][0]['content'][:300])
print('\n--- CoT system ---')
print(c_row['prompt'][0]['content'][:300])
print('\n--- User (both) ---')
print(next(m['content'] for m in t_row['prompt'] if m['role'] == 'user')[:500])
print('\n--- Ground truth ---')
print(t_row['reward_model']['ground_truth'][:300])

User message equal : True
Ground truth equal : True
Pool equal         : True

--- TIR system ---
You are an expert physics problem solver.
You have access to a Python interpreter. Use it when it helps — for numerical computation, symbolic algebra, or unit conversion. You are not required to use it.
Only call the tool if the code performs actual computation — numerical evaluation, symbolic solvi

--- CoT system ---
You are an expert physics problem solver.
Solve the problem step by step. Show your reasoning concisely.

End with your final answer as \boxed{<value>}.


--- User (both) ---
A certain atom's ${}^{3} \mathrm{P}_{2}$ energy level is found to split into five sublevels, with the ratio of the intervals between adjacent sublevels being $9: 7: 5: 3$. Using the interval rule, determine the nuclear spin quantum number $I$ of this atom and the total atomic angular momentum quantum number $F$ corresponding to each sublevel. Find the nuclear spin quantum number $I$.

--- Ground truth -

## 10. Prompt length distribution

Character count of user message (proxy for token count). Useful to spot outliers.

In [11]:
user_len = tir['train']['prompt'].apply(lambda msgs: len(next(m['content'] for m in msgs if m['role'] == 'user')))
gt_len = tir['train']['reward_model'].apply(lambda x: len(x['ground_truth']))

print('User message length (chars):')
print(user_len.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(0))
print('\nGround-truth length (chars):')
print(gt_len.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(0))

# Longest and shortest user message
print('\nShortest user message:')
print(tir['train']['prompt'].iloc[user_len.idxmin()])
print('\nLongest user message (first 500 chars):')
longest_idx = user_len.idxmax()
msgs = tir['train']['prompt'].iloc[longest_idx]
print(next(m['content'] for m in msgs if m['role'] == 'user')[:500])

User message length (chars):
count    107977.0
mean        399.0
std         228.0
min          12.0
50%         351.0
90%         642.0
95%         821.0
99%        1304.0
max       11570.0
Name: prompt, dtype: float64

Ground-truth length (chars):
count    107977.0
mean         27.0
std          46.0
min           1.0
50%          11.0
90%          71.0
95%         106.0
99%         210.0
max        2029.0
Name: reward_model, dtype: float64

Shortest user message:
[{'content': 'You are an expert physics problem solver.\nYou have access to a Python interpreter. Use it when it helps — for numerical computation, symbolic algebra, or unit conversion. You are not required to use it.\nOnly call the tool if the code performs actual computation — numerical evaluation, symbolic solving, or unit conversion. Do not use it to print a formula or expression you derived in text.\n\nAllowed packages: numpy, scipy, sympy, pint, math, cmath, statistics, fractions, decimal, itertools, functools, collec

## 11. Sanity checks (assert-style)

In [12]:
checks = []

# Row counts
checks.append(('TIR train  == 107977', len(tir['train']) == 107977))
checks.append(('TIR val    == 2200',   len(tir['validation']) == 2200))
checks.append(('TIR test   == 2200',   len(tir['test']) == 2200))

# CoT has same sizes as TIR
for s in ['train', 'validation', 'test']:
    checks.append((f'TIR/CoT {s} equal size', len(tir[s]) == len(cot[s])))

# Exactly one system prompt per dataset
checks.append(('TIR has 1 system prompt', len(unique_sys_prompts(tir['train'])) == 1))
checks.append(('CoT has 1 system prompt', len(unique_sys_prompts(cot['train'])) == 1))

# Every row has exactly 2 messages (system + user)
checks.append(('TIR all rows have 2 msgs', (tir['train']['prompt'].apply(len) == 2).all()))
checks.append(('CoT all rows have 2 msgs', (cot['train']['prompt'].apply(len) == 2).all()))

# Every row has non-empty user message
user_lens = tir['train']['prompt'].apply(lambda msgs: len(next(m['content'] for m in msgs if m['role'] == 'user')))
checks.append(('TIR user msgs non-empty', (user_lens > 0).all()))

# Every row has non-empty ground truth
gt_lens = tir['train']['reward_model'].apply(lambda x: len(x['ground_truth']))
checks.append(('TIR ground truths non-empty', (gt_lens > 0).all()))

# Contamination
sources_all = set()
for split, df in tir.items():
    sources_all |= set(df['extra_info'].apply(lambda x: x.get('source')).dropna().unique())
checks.append(('No contamination (TIR)', len(sources_all & DROP) == 0))

# CoT system prompt contains no tool keywords
cot_sys_lower = cot_sys[0].lower()
checks.append(('CoT sys has no "tool"',   'tool'   not in cot_sys_lower))
checks.append(('CoT sys has no "python"', 'python' not in cot_sys_lower))
checks.append(('CoT sys has no "code"',   'code'   not in cot_sys_lower))
checks.append(('CoT sys has "boxed"',     'boxed'  in cot_sys_lower))

# TIR system prompt DOES mention tool/python/code
tir_sys_lower = tir_sys[0].lower()
checks.append(('TIR sys mentions python/tool', ('python' in tir_sys_lower) or ('tool' in tir_sys_lower)))

for label, ok in checks:
    print(f'  [{"PASS" if ok else "FAIL"}] {label}')
print(f'\n{sum(ok for _, ok in checks)}/{len(checks)} checks passed')

  [PASS] TIR train  == 107977
  [PASS] TIR val    == 2200
  [PASS] TIR test   == 2200
  [PASS] TIR/CoT train equal size
  [PASS] TIR/CoT validation equal size
  [PASS] TIR/CoT test equal size
  [PASS] TIR has 1 system prompt
  [PASS] CoT has 1 system prompt
  [PASS] TIR all rows have 2 msgs
  [PASS] CoT all rows have 2 msgs
  [PASS] TIR user msgs non-empty
  [PASS] TIR ground truths non-empty
  [PASS] No contamination (TIR)
  [PASS] CoT sys has no "tool"
  [PASS] CoT sys has no "python"
  [PASS] CoT sys has no "code"
  [PASS] CoT sys has "boxed"
  [PASS] TIR sys mentions python/tool

18/18 checks passed
